# GWAS Meta-analysis using METAL
**Phenotypes:** Congenital Malformations of the Great Arteries & Septal Defects  
**Cohorts:** FinnGen R12, UK Biobank (GWAS Catalog)  
**SNP filter:** ADHD PGC 2022 reference panel  

## 1. Environment Setup

In [ ]:
install.packages("R.utils", quiet = TRUE)
library(data.table)
library(dplyr)

In [ ]:
# Wrapper for shell commands that captures and prints output
shell_exec <- function(command, ...) {
  result <- system(command, intern = TRUE, ...)
  cat(paste(result, collapse = "\n"), "\n")
  invisible(result)
}

## 2. Install METAL

In [ ]:
system("apt-get update -qq && apt-get install -y build-essential cmake git wget -qq")

if (!dir.exists("METAL")) {
  system("git clone https://github.com/statgen/METAL.git METAL")
}

system(
  "mkdir -p METAL/build && cd METAL/build && cmake .. >/dev/null && make -j2 >/dev/null",
  wait = TRUE
)

In [ ]:
# Resolve METAL binary path
metal_candidate_paths <- c(
  "METAL/build/metal",
  "METAL/metal",
  "METAL/build/metal/metal"
)

metal_path <- Sys.which("metal")
if (metal_path == "") {
  metal_path <- metal_candidate_paths[file.exists(metal_candidate_paths)][1]
}
if (is.na(metal_path) || metal_path == "") {
  stop("METAL binary not found. Check build logs in METAL/build.")
}

cat("METAL binary found at:", metal_path, "\n")

## 3. Download ADHD PGC Reference SNP List
The ADHD 2022 PGC summary statistics are used as a SNP reference set to filter the large UKBB files to a tractable size.

In [ ]:
shell_exec(
  'wget --user-agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0" \
  -O adhdPGC.zip "https://figshare.com/ndownloader/articles/22564390/versions/1"'
)
shell_exec("unzip -o /content/adhdPGC.zip")

snp_reference <- fread(
  "/content/ADHD2022_iPSYCH_deCODE_PGC.meta.gz",
  select = "SNP"
)
fwrite(
  data.table(SNP = snp_reference$SNP),
  "snps_to_keep.txt",
  col.names = FALSE
)
cat("Reference SNP list written:", nrow(snp_reference), "SNPs\n")
rm(snp_reference)

---
## 4. Meta-analysis: Congenital Malformations of the Great Arteries
### 4.1 FinnGen R12 — Great Arteries

In [ ]:
shell_exec(
  "curl https://storage.googleapis.com/finngen-public-data-r12/summary_stats/release/\
finngen_R12_Q17_CONGEN_MALFO_GREAT_ARTERIES.gz \
  -o finngen_R12_Q17_CONGEN_MALFO_GREAT_ARTERIES.gz"
)

# FinnGen R12 sample sizes: 448,451 controls + 687 cases
N_FINNGEN_CONTROLS <- 448451L
N_FINNGEN_CASES_GA <- 687L

finngen_great_arteries <- fread("/content/finngen_R12_Q17_CONGEN_MALFO_GREAT_ARTERIES.gz")
finngen_great_arteries[, N := N_FINNGEN_CONTROLS + N_FINNGEN_CASES_GA]

finngen_great_arteries <- finngen_great_arteries %>%
  select(
    SNP   = rsids,
    CHR   = `#chrom`,
    A1    = alt,
    A2    = ref,
    N,
    B     = beta,
    SE    = sebeta,
    P     = pval
  )

fwrite(finngen_great_arteries, file = "chd_summaries.FinnGen.txt", sep = "\t")
cat("FinnGen Great Arteries written:", nrow(finngen_great_arteries), "variants\n")
rm(finngen_great_arteries)

### 4.2 UK Biobank — Great Arteries (GCST90474263)

In [ ]:
shell_exec(
  "curl https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/\
GCST90474001-GCST90475000/GCST90474263/harmonised/GCST90474263.h.tsv.gz \
  -o GCST90474260.h.tsv.gz"
)

# Extract columns of interest to reduce memory footprint before loading into R
# Columns: 1=chrom, 3=effect_allele, 4=other_allele, 5=beta, 6=se, 8=p_value, 9=rsid, 12=n
system(
  "zcat /content/GCST90474260.h.tsv.gz | awk -F'\t' '{print $1,$3,$4,$5,$6,$8,$9,$12}' \
  OFS='\t' | gzip > reduced_file_ukbb.tsv.gz"
)
system("gunzip -c /content/reduced_file_ukbb.tsv.gz > reduced_file_ukbb.tsv")

# Filter to reference SNP list using awk for efficiency
system(
  "awk 'NR==1{print; next} FNR==NR{a[$1]; next} $7 in a' \
  snps_to_keep.txt /content/reduced_file_ukbb.tsv > filtered_file_ukbb.tsv"
)

ukbb_great_arteries <- fread("filtered_file_ukbb.tsv")
colnames(ukbb_great_arteries) <- c(
  "chromosome", "effect_allele", "other_allele",
  "beta", "standard_error", "p_value", "rsid", "n"
)

fwrite(ukbb_great_arteries, file = "chd_summaries.UKBB.txt", sep = "\t")
cat("UKBB Great Arteries written:", nrow(ukbb_great_arteries), "variants\n")
rm(ukbb_great_arteries)

### 4.3 Run METAL — Great Arteries

In [ ]:
metal_script_great_arteries <- c(
  "SCHEME STDERR",
  "",
  "# FinnGen R12 — Great Arteries",
  "MARKER SNP",
  "ALLELE A1 A2",
  "EFFECT B",
  "STDERR SE",
  "PVALUE P",
  "WEIGHT N",
  "PROCESS /content/chd_summaries.FinnGen.txt",
  "",
  "# UK Biobank — Great Arteries (GCST90474263)",
  "MARKER rsid",
  "ALLELE effect_allele other_allele",
  "EFFECT beta",
  "STDERR standard_error",
  "PVALUE p_value",
  "WEIGHT n",
  "PROCESS /content/chd_summaries.UKBB.txt",
  "",
  "ANALYZE"
)
writeLines(metal_script_great_arteries, con = "metal_script_great_arteries.txt")

In [ ]:
log_stdout_ga <- "metal_great_arteries.log"
log_stderr_ga <- "metal_great_arteries.err"

cmd_ga <- sprintf(
  "%s metal_script_great_arteries.txt > %s 2> %s",
  shQuote(metal_path), log_stdout_ga, log_stderr_ga
)
system(cmd_ga, wait = TRUE)

cat("=== METAL STDOUT (last 50 lines) ===\n")
system(sprintf("tail -n 50 %s || true", log_stdout_ga))
cat("\n=== METAL STDERR (last 50 lines) ===\n")
system(sprintf("tail -n 50 %s || true", log_stderr_ga))

shell_exec("mv METAANALYSIS1.TBL METAANALYSIS1_Great.TBL")
cat("Output file: METAANALYSIS1_Great.TBL\n")

---
## 5. Meta-analysis: Septal Defects
### 5.1 FinnGen R12 — Septal Defects

In [ ]:
shell_exec(
  "curl https://storage.googleapis.com/finngen-public-data-r12/summary_stats/release/\
finngen_R12_Q17_SEPTA_DEFEC.gz \
  -o finngen_R12_Q17_SEPTA_DEFEC.gz"
)

# FinnGen R12 sample sizes: 448,451 controls + 687 cases
N_FINNGEN_CASES_SD <- 687L

finngen_septal <- fread("/content/finngen_R12_Q17_SEPTA_DEFEC.gz")
finngen_septal[, N := N_FINNGEN_CONTROLS + N_FINNGEN_CASES_SD]

finngen_septal <- finngen_septal %>%
  select(
    SNP   = rsids,
    CHR   = `#chrom`,
    A1    = alt,
    A2    = ref,
    N,
    B     = beta,
    SE    = sebeta,
    P     = pval
  )

fwrite(finngen_septal, file = "chd_summaries.FinnGen.txt", sep = "\t")
cat("FinnGen Septal Defects written:", nrow(finngen_septal), "variants\n")
rm(finngen_septal)

### 5.2 UK Biobank — Septal Defects (GCST90474260)

In [ ]:
shell_exec(
  "curl https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/\
GCST90474001-GCST90475000/GCST90474260/harmonised/GCST90474260.h.tsv.gz \
  -o GCST90474260.h.tsv.gz"
)

# Extract columns of interest to reduce memory footprint before loading into R
# Columns: 1=chrom, 3=effect_allele, 4=other_allele, 5=beta, 6=se, 8=p_value, 9=rsid, 12=n
system(
  "zcat /content/GCST90474260.h.tsv.gz | awk -F'\t' '{print $1,$3,$4,$5,$6,$8,$9,$12}' \
  OFS='\t' | gzip > reduced_file_ukbb.tsv.gz"
)
system("gunzip -c /content/reduced_file_ukbb.tsv.gz > reduced_file_ukbb.tsv")

# Filter to reference SNP list using awk for efficiency
system(
  "awk 'NR==1{print; next} FNR==NR{a[$1]; next} $7 in a' \
  snps_to_keep.txt /content/reduced_file_ukbb.tsv > filtered_file_ukbb.tsv"
)

ukbb_septal <- fread("filtered_file_ukbb.tsv")
colnames(ukbb_septal) <- c(
  "chromosome", "effect_allele", "other_allele",
  "beta", "standard_error", "p_value", "rsid", "n"
)

fwrite(ukbb_septal, file = "chd_summaries.UKBB.txt", sep = "\t")
cat("UKBB Septal Defects written:", nrow(ukbb_septal), "variants\n")
rm(ukbb_septal)

### 5.3 Run METAL — Septal Defects

In [ ]:
metal_script_septal <- c(
  "SCHEME STDERR",
  "",
  "# FinnGen R12 — Septal Defects",
  "MARKER SNP",
  "ALLELE A1 A2",
  "EFFECT B",
  "STDERR SE",
  "PVALUE P",
  "WEIGHT N",
  "PROCESS /content/chd_summaries.FinnGen.txt",
  "",
  "# UK Biobank — Septal Defects (GCST90474260)",
  "MARKER rsid",
  "ALLELE effect_allele other_allele",
  "EFFECT beta",
  "STDERR standard_error",
  "PVALUE p_value",
  "WEIGHT n",
  "PROCESS /content/chd_summaries.UKBB.txt",
  "",
  "ANALYZE"
)
writeLines(metal_script_septal, con = "metal_script_septal.txt")

In [ ]:
log_stdout_sd <- "metal_septal.log"
log_stderr_sd <- "metal_septal.err"

cmd_sd <- sprintf(
  "%s metal_script_septal.txt > %s 2> %s",
  shQuote(metal_path), log_stdout_sd, log_stderr_sd
)
system(cmd_sd, wait = TRUE)

cat("=== METAL STDOUT (last 50 lines) ===\n")
system(sprintf("tail -n 50 %s || true", log_stdout_sd))
cat("\n=== METAL STDERR (last 50 lines) ===\n")
system(sprintf("tail -n 50 %s || true", log_stderr_sd))

shell_exec("mv METAANALYSIS1.TBL METAANALYSIS1_Septal.TBL")
cat("Output file: METAANALYSIS1_Septal.TBL\n")

---
## 6. Verify Outputs

In [ ]:
output_files <- list.files(pattern = "METAANALYSIS1_(Great|Septal)\\.TBL$")
cat("Meta-analysis output files:\n")
print(output_files)